# Vision Language Models (VLMs)

## What Are VLMs?
Vision Language Models process both images and text, enabling tasks like image captioning, visual QA, document understanding, and image-based reasoning.

## Architecture Patterns

### 1. Adapter-based (LLaVA style)
```
Image → Vision Encoder (CLIP/SigLIP) → Projection Layer (MLP) → LLM
Text  ──────────────────────────────────────────────────────→ LLM
```

### 2. Cross-Attention (Flamingo style)
```
Image → Vision Encoder → Cross-Attention Layers → LLM
Text  ─────────────────────────────────────────→ LLM
```

### 3. Q-Former (BLIP-2 style)
```
Image → Frozen Vision Encoder → Q-Former (learnable queries) → Frozen LLM
```

### 4. Native Multimodal (Gemini, GPT-4o)
```
All modalities processed jointly from pretraining
```

## VLM Landscape

| Model | Params | Creator | Speciality |
|-------|--------|---------|------------|
| GPT-4o | ~200B | OpenAI | Best overall |
| Claude 3.5 Sonnet | Unknown | Anthropic | Document understanding |
| Gemini 1.5 Pro | Unknown | Google | Video, long context |
| LLaVA-NeXT | 7-72B | HaotianLiu | Open source pioneer |
| InternVL2 | 2-108B | Shanghai AI | State-of-art open |
| Qwen2-VL | 2-72B | Alibaba | Strong document OCR |
| PaliGemma 2 | 3-28B | Google | Open, versatile |
| Phi-3.5-Vision | 4.2B | Microsoft | Edge VLM |
| BLIP-2 | 7B | Salesforce | Q-Former architecture |
| Florence-2 | 0.2-0.8B | Microsoft | Detection + captioning |

In [1]:
# --- GPT-4o Vision ---
import base64
from openai import OpenAI

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def query_gpt4o_vision(image_path, question):
    client = OpenAI()
    b64 = encode_image(image_path)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                {"type": "text", "text": question}
            ]
        }]
    )
    return response.choices[0].message.content

# Also supports URL:
def query_gpt4o_url(image_url, question):
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": question}
            ]
        }]
    )
    return response.choices[0].message.content

print('GPT-4o vision functions defined.')

GPT-4o vision functions defined.


In [2]:
# --- Claude Vision ---
import anthropic
import base64
from pathlib import Path

def query_claude_vision(image_path, question, model="claude-sonnet-4-6"):
    client = anthropic.Anthropic()
    image_data = base64.b64encode(Path(image_path).read_bytes()).decode()
    ext = Path(image_path).suffix.lstrip('.')
    media_type = f"image/{ext}" if ext != 'jpg' else "image/jpeg"

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "base64", "media_type": media_type, "data": image_data}
                },
                {"type": "text", "text": question}
            ]
        }]
    )
    return response.content[0].text

print('Claude vision function defined.')

Claude vision function defined.


In [3]:
# --- Open-source VLM with Transformers: InternVL2 ---
INTERNVL_CODE = '''
from transformers import AutoModel, AutoTokenizer
from PIL import Image
import torch

model = AutoModel.from_pretrained(
    "OpenGVLab/InternVL2-8B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained("OpenGVLab/InternVL2-8B", trust_remote_code=True)

image = Image.open("image.jpg").convert("RGB")
question = "<image>\nWhat do you see in this image?"

response = model.chat(tokenizer, image, question, generation_config={"max_new_tokens": 512})
print(response)
'''
print(INTERNVL_CODE)


from transformers import AutoModel, AutoTokenizer
from PIL import Image
import torch

model = AutoModel.from_pretrained(
    "OpenGVLab/InternVL2-8B",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained("OpenGVLab/InternVL2-8B", trust_remote_code=True)

image = Image.open("image.jpg").convert("RGB")
question = "<image>
What do you see in this image?"

response = model.chat(tokenizer, image, question, generation_config={"max_new_tokens": 512})
print(response)



In [4]:
# --- PaliGemma: Google's open VLM ---
PALIGEMMA_CODE = '''
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from PIL import Image
import torch

model_id = "google/paligemma2-3b-pt-224"
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16)
processor = PaliGemmaProcessor.from_pretrained(model_id)

image = Image.open("image.jpg")
inputs = processor(text="Describe this image:", images=image, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200)

print(processor.decode(output[0], skip_special_tokens=True))
'''
print(PALIGEMMA_CODE)


from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from PIL import Image
import torch

model_id = "google/paligemma2-3b-pt-224"
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16)
processor = PaliGemmaProcessor.from_pretrained(model_id)

image = Image.open("image.jpg")
inputs = processor(text="Describe this image:", images=image, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200)

print(processor.decode(output[0], skip_special_tokens=True))



In [5]:
# --- Florence-2: detection + captioning ---
FLORENCE_CODE = '''
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import torch

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

image = Image.open("image.jpg")

# Tasks: <CAPTION>, <DETAILED_CAPTION>, <OD>, <DENSE_REGION_CAPTION>,
#        <OPEN_VOCABULARY_DETECTION>, <SEGMENTATION>, <OCR>

for task in ["<CAPTION>", "<OD>", "<OCR>"]:
    inputs = processor(text=task, images=image, return_tensors="pt")
    generated = model.generate(**inputs, max_new_tokens=256)
    result = processor.batch_decode(generated, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(result, task=task, image_size=image.size)
    print(f"{task}: {parsed}")
'''
print(FLORENCE_CODE)


from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import torch

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

image = Image.open("image.jpg")

# Tasks: <CAPTION>, <DETAILED_CAPTION>, <OD>, <DENSE_REGION_CAPTION>,
#        <OPEN_VOCABULARY_DETECTION>, <SEGMENTATION>, <OCR>

for task in ["<CAPTION>", "<OD>", "<OCR>"]:
    inputs = processor(text=task, images=image, return_tensors="pt")
    generated = model.generate(**inputs, max_new_tokens=256)
    result = processor.batch_decode(generated, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(result, task=task, image_size=image.size)
    print(f"{task}: {parsed}")



## Additional Learning Resources

### Papers
- [LLaVA](https://arxiv.org/abs/2304.08485) Liu et al., 2023
- [BLIP-2](https://arxiv.org/abs/2301.12597) Li et al., 2023
- [Flamingo](https://arxiv.org/abs/2204.14198) Alayrac et al., 2022
- [CLIP](https://arxiv.org/abs/2103.00020) Radford et al., 2021
- [PaliGemma 2](https://arxiv.org/abs/2412.03555)
- [Florence-2](https://arxiv.org/abs/2311.06242)
- [Qwen2-VL](https://arxiv.org/abs/2409.12191)

### Resources
- [InternVL GitHub](https://github.com/OpenGVLab/InternVL)
- [Hugging Face VLM Models](https://huggingface.co/models?pipeline_tag=image-text-to-text)
- [OpenAI Vision Guide](https://platform.openai.com/docs/guides/vision)
- [Anthropic Vision Guide](https://docs.anthropic.com/en/docs/vision)